# SHIPP — Demo data readiness

The demo must produce at least one real candidate match in Abu Dhabi.
Current problem: `demo-listing-001` (25.2854, 51.531) and `demo-listing-002` (25.3, 51.52) are in **Doha, Qatar**.

This notebook:
1. Shows the current demo rows in Silver.
2. Prints the Lakebase `UPDATE` SQL that moves them to Abu Dhabi (matches the Figma demo: desk on Al Reem Island, requester in Al Khalidiyah ≈ 6.4 km).
3. After you run the SQL and rerun Silver, checks every precondition for a candidate pair.

Coordinates are approximate area centres, good enough for routing.

In [0]:
%run ../common/shipp_silver_lib

## Step 1 — Current demo rows in Silver

In [0]:
display(
    spark.table(TABLES["silver_listings"])
    .select("listing_id", "donor_id", "title", "category", "condition", "status",
            "latitude", "longitude", "available_from", "available_until")
    .orderBy("listing_id")
)
display(
    spark.table(TABLES["silver_requests"])
    .select("request_id", "requester_id", "request_text", "category", "status",
            "latitude", "longitude", "need_by_date")
    .orderBy("request_id")
)

## Step 2 — Run this SQL in the **Lakebase SQL editor**

Then run `development/10_silver_listings_dev` and `development/11_silver_requests_dev`
(or `pipeline/90_run_pipeline`), and continue with Step 3.

In [0]:
print("""UPDATE shipp.listings
SET location = 'Al Reem Island, Abu Dhabi', latitude = 24.4989, longitude = 54.4057,
    updated_at = CURRENT_TIMESTAMP
WHERE listing_id = 'demo-listing-001';

UPDATE shipp.listings
SET location = 'Al Khalidiyah, Abu Dhabi', latitude = 24.4697, longitude = 54.3494,
    updated_at = CURRENT_TIMESTAMP
WHERE listing_id = 'demo-listing-002';

UPDATE shipp.requests
SET location = 'Al Khalidiyah, Abu Dhabi', latitude = 24.4700, longitude = 54.3490,
    category = 'FURNITURE',
    need_by_date = CURRENT_TIMESTAMP + INTERVAL '30 days',
    updated_at = CURRENT_TIMESTAMP
WHERE request_id = 'demo-request-001';

SELECT listing_id, location, latitude, longitude FROM shipp.listings WHERE listing_id LIKE 'demo-%';
SELECT request_id, location, latitude, longitude, category, need_by_date, status
FROM shipp.requests WHERE request_id LIKE 'demo-%';""")

## Step 3 — Preconditions for at least one candidate pair (run after Silver is refreshed)

In [0]:
# Bounding box around Abu Dhabi + Dubai (demo service area)
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 23.5, 25.6, 53.5, 55.8

listings = spark.table(TABLES["silver_listings"])
requests_df = spark.table(TABLES["silver_requests"])

outside = (
    listings.filter(~(F.col("latitude").between(LAT_MIN, LAT_MAX) & F.col("longitude").between(LON_MIN, LON_MAX)))
    .select("listing_id", "latitude", "longitude").collect()
    + requests_df.filter(~(F.col("latitude").between(LAT_MIN, LAT_MAX) & F.col("longitude").between(LON_MIN, LON_MAX)))
    .select("request_id", "latitude", "longitude").collect()
)
available = listings.filter(F.col("status") == "AVAILABLE").count()
open_requests = requests_df.filter(
    F.col("status").isin("OPEN", "MATCHES_AVAILABLE") & (F.col("need_by_date") >= now_ntz())
).count()
shared_categories = (
    listings.filter(F.col("status") == "AVAILABLE").select("category").distinct()
    .intersect(requests_df.select("category").distinct()).count()
)

print("Rows outside the Abu Dhabi/Dubai demo area:", outside)
print("AVAILABLE listings:                         ", available)
print("OPEN requests with future need_by_date:     ", open_requests)
print("Categories shared by listings and requests: ", shared_categories)

assert not outside, "FAILED: some demo coordinates are outside the service area — run the Step 2 SQL."
assert available > 0, "FAILED: no AVAILABLE listing."
assert open_requests > 0, "FAILED: no OPEN request with a future need_by_date (check status values and need_by_date)."
assert shared_categories > 0, "FAILED: no category is shared by an available listing and a request."
print("PASS — demo data can produce candidate pairs.")